# Factor-Adaptive Mean Reversion Extended

An extended adaptive mean-reversion backtest.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
FUSION ADAPTIVE MULTI-REGIME STRATEGY (FAMRS) - FIXED VERSION
==============================================================

CRITICAL FIXES IMPLEMENTED:
--------------------------
1. EXECUTION TIMING: Standardized to Close[T-1] signal â†’ Open[T] execution
   - Features calculated using data up to Close[T-1]
   - Signals generated at Close[T-1]
   - Execution at Open[T]
   - Returns: Open[T+1]/Open[T] - 1
   - Portfolio valued at Open on execution days, Close during holding

2. TRUE LEVERAGE: Proper leverage accounting
   - Negative cash allowed when leveraged
   - Daily financing costs: interest = max(0, gross_exposure - equity) Ã— rate/252
   - Hard cap at MAX_LEVERAGE (default 1.5x)

3. ADJUSTED CLOSE: All calculations use adjusted prices
   - Returns, volatility, momentum use adjusted close
   - Eliminates distortions from splits/dividends

4. BENCHMARK ALIGNMENT: Same timing as strategy
   - Open-to-Open returns on execution dates
   - Fair apples-to-apples comparison

5. STOP-LOSS TIMING: Consistent with execution
   - Checked at Close[T]
   - Exit at Open[T+1]
   - Includes slippage in cost model

STRATEGIC CONCEPT:
-----------------
Multi-asset consensus regime detection with graduated exposure system.
Combines best practices from documented strategies with proper execution mechanics.
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime, timedelta
from typing import Dict, Tuple, List, Optional
from dataclasses import dataclass, field
from sklearn.preprocessing import StandardScaler
from scipy.stats import t as student_t
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform
from sklearn.covariance import LedoitWolf

warnings.filterwarnings("ignore")


@dataclass
class StrategyConfig:
    """Centralized configuration with all parameters exposed as inputs."""

    BACKTEST_START_DATE: str = "2013-01-01"
    BACKTEST_END_DATE: str = "2018-12-31"
    INITIAL_CAPITAL: float = 100000.0
    BENCHMARK_TICKER: str = "URTH"
    MACRO_ASSETS: List[str] = field(default_factory=lambda: ["SPY", "TLT", "HYG"])
    VIX_TICKER: str = "^VIX"
    MACRO_WEIGHTS: Optional[Dict[str, float]] = None
    HIGH_BETA_ASSETS: List[str] = field(
        default_factory=lambda: ["QQQ", "XLK", "VGT", "XLI", "XLY", "EEM", "IWM", "XLF"]
    )
    BALANCED_ASSETS: List[str] = field(
        default_factory=lambda: ["SPY", "QQQ", "XLK", "VYM", "SCHD", "DGRO", "VUG", "XLV"]
    )
    NEUTRAL_ASSETS: List[str] = field(
        default_factory=lambda: ["SPY", "QQQ", "VYM", "USMV", "XLU", "XLP", "XLV", "VMOT"]
    )
    DEFENSIVE_ASSETS: List[str] = field(
        default_factory=lambda: ["USMV", "SPLV", "XLU", "XLP", "TLT", "IEF", "XLV", "VYM"]
    )
    SAFE_HAVEN_ASSETS: List[str] = field(
        default_factory=lambda: ["TLT", "IEF", "XLU", "XLP", "GLD", "AGG", "SHY"]
    )
    HEDGE_TICKER: str = "SH"
    HMM_N_STATES: int = 2
    HMM_N_ITER: int = 150
    HMM_RANDOM_STATE: int = 42
    HMM_CONVERGENCE_TOL: float = 0.0001
    STUDENT_T_DF_INIT: float = 5.0
    STUDENT_T_DF_MIN: float = 2.1
    STUDENT_T_DF_MAX: float = 30.0
    VOL_WINDOW: int = 20
    MOMENTUM_WINDOWS: List[int] = field(default_factory=lambda: [5, 10, 21, 63])
    MOMENTUM_WEIGHTS: List[float] = field(default_factory=lambda: [0.4, 0.3, 0.2, 0.1])
    MIN_TRAINING_DAYS: int = 252
    REFIT_FREQUENCY: int = 63
    REBALANCE_FREQUENCY: int = 21
    STRONG_RISK_ON_THRESHOLD: float = 0.7
    MODERATE_RISK_ON_THRESHOLD: float = 0.52
    MODERATE_RISK_OFF_THRESHOLD: float = 0.45
    STRONG_RISK_OFF_THRESHOLD: float = 0.35
    STRONG_RISK_ON_EXPOSURE: float = 1.2
    MODERATE_RISK_ON_EXPOSURE: float = 1.0
    NEUTRAL_EXPOSURE: float = 0.8
    MODERATE_RISK_OFF_EXPOSURE: float = 0.6
    STRONG_RISK_OFF_LONG_EXPOSURE: float = 0.4
    STRONG_RISK_OFF_HEDGE_EXPOSURE: float = 0.1
    STRONG_RISK_ON_MAX_POSITIONS: int = 8
    MODERATE_RISK_ON_MAX_POSITIONS: int = 10
    NEUTRAL_MAX_POSITIONS: int = 12
    MODERATE_RISK_OFF_MAX_POSITIONS: int = 10
    STRONG_RISK_OFF_MAX_POSITIONS: int = 6
    STRONG_RISK_ON_MIN_WEIGHT: float = 0.08
    STRONG_RISK_ON_MAX_WEIGHT: float = 0.25
    MODERATE_RISK_ON_MIN_WEIGHT: float = 0.08
    MODERATE_RISK_ON_MAX_WEIGHT: float = 0.2
    NEUTRAL_MIN_WEIGHT: float = 0.06
    NEUTRAL_MAX_WEIGHT: float = 0.15
    DEFENSIVE_MIN_WEIGHT: float = 0.08
    DEFENSIVE_MAX_WEIGHT: float = 0.12
    STOP_LOSS_ENABLED: bool = True
    STOP_LOSS_METHOD: str = "cvar95"
    CVAR_CONFIDENCE: float = 0.95
    CVAR_LOOKBACK: int = 60
    STOP_LOSS_BUFFER: float = 0.8
    STOP_LOSS_SLIPPAGE: float = 0.002
    MIN_POSITION_CHANGE: float = 0.03
    COMMISSION_RATE: float = 0.0005
    BID_ASK_SPREAD: float = 0.0003
    MARKET_IMPACT: float = 0.0002
    MIN_COMMISSION: float = 1.0
    MAX_LEVERAGE: float = 1.5
    MARGIN_RATE_ANNUAL: float = 0.06
    RISK_FREE_RATE: float = 0.02

    def __post_init__(self):
        """Initialize default weights if not provided."""
        if self.MACRO_WEIGHTS is None:
            self.MACRO_WEIGHTS = {"SPY": 0.4, "TLT": 0.3, "HYG": 0.2, "^VIX": 0.1}


class DataFetcher:
    """
    Fetches historical data from Yahoo Finance.

    CRITICAL FIX: Now fetches adjusted close prices to handle splits/dividends.
    """

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        """
        Fetch historical price data with adjusted close.

        IMPROVEMENT: Uses adjusted close for all price calculations to avoid
        distortions from splits and dividend distributions.
        """
        original_ticker = ticker
        if ticker.startswith("^"):
            ticker_variants = [ticker[1:], ticker]
        else:
            ticker_variants = [ticker]
        last_error = None
        for ticker_variant in ticker_variants:
            try:
                response_prices = {
                    "historical": download_price_records(ticker_variant, start_date, end_date)
                }
                data = response_prices
                if "historical" not in data or len(data["historical"]) < 100:
                    last_error = f"Insufficient data: {len(data.get('historical', []))} days"
                    continue
                df = pd.DataFrame(data["historical"])
                df["date"] = pd.to_datetime(df["date"])
                df = df.sort_values("date").reset_index(drop=True)
                df = df.rename(
                    columns={
                        "date": "Date",
                        "open": "Open",
                        "high": "High",
                        "low": "Low",
                        "close": "Close",
                        "adjClose": "Adj_Close",
                        "volume": "Volume",
                    }
                )
                if "Adj_Close" not in df.columns or df["Adj_Close"].isna().all():
                    print(f"  ! {original_ticker}: No adjusted close, using raw close")
                    df["Adj_Close"] = df["Close"]
                df["adj_factor"] = df["Adj_Close"] / df["Close"]
                df["Adj_Open"] = df["Open"] * df["adj_factor"]
                df["Adj_High"] = df["High"] * df["adj_factor"]
                df["Adj_Low"] = df["Low"] * df["adj_factor"]
                null_pct = df["Adj_Close"].isnull().sum() / len(df)
                if null_pct > 0.05:
                    last_error = f"Excessive missing data: {null_pct:.1%}"
                    continue
                print(f"  âœ“ {original_ticker:6s}: {len(df):4d} days (adjusted prices)")
                return df
            except Exception as e:
                last_error = str(e)
                continue
        print(f"  âœ— Error fetching {original_ticker}: {last_error}")
        raise ValueError(f"Failed to fetch {original_ticker}: {last_error}")

    def fetch_all_data(self, config: StrategyConfig) -> Dict[str, pd.DataFrame]:
        """Fetch all required data with buffer for feature calculation."""
        print("\n" + "=" * 80)
        print("DATA ACQUISITION")
        print("=" * 80)
        print(f"Period: {config.BACKTEST_START_DATE} to {config.BACKTEST_END_DATE}")
        buffer_start = (
            datetime.strptime(config.BACKTEST_START_DATE, "%Y-%m-%d")
            - timedelta(days=config.MIN_TRAINING_DAYS + 100)
        ).strftime("%Y-%m-%d")
        print(f"Fetching from: {buffer_start} (includes buffer)")
        all_tickers = list(
            set(
                config.MACRO_ASSETS
                + [config.VIX_TICKER]
                + config.HIGH_BETA_ASSETS
                + config.BALANCED_ASSETS
                + config.NEUTRAL_ASSETS
                + config.DEFENSIVE_ASSETS
                + config.SAFE_HAVEN_ASSETS
                + [config.HEDGE_TICKER, config.BENCHMARK_TICKER]
            )
        )
        print(f"Total unique tickers to fetch: {len(all_tickers)}")
        data_dict = {}
        failed_tickers = []
        for ticker in all_tickers:
            try:
                df = self.fetch_historical_data(ticker, buffer_start, config.BACKTEST_END_DATE)
                data_dict[ticker] = df
            except Exception as e:
                failed_tickers.append(ticker)
                continue
        print(f"\nâœ“ Successfully fetched: {len(data_dict)} tickers")
        if failed_tickers:
            print(
                f"âœ— Failed to fetch: {len(failed_tickers)} tickers: {', '.join(failed_tickers)}"
            )
        required_macro = [t for t in config.MACRO_ASSETS if t in data_dict]
        if len(required_macro) < 2:
            raise ValueError(f"Insufficient macro assets: only {required_macro} available")
        return data_dict


class StudentTHMM:
    """Student-t HMM for robust regime detection - handles fat tails."""

    def __init__(
        self,
        n_states: int = 2,
        n_iter: int = 150,
        tol: float = 0.0001,
        random_state: int = 42,
        df_init: float = 5.0,
        df_min: float = 2.1,
        df_max: float = 30.0,
    ):
        self.n_states = n_states
        self.n_iter = n_iter
        self.tol = tol
        self.random_state = random_state
        self.df_init = df_init
        self.df_min = df_min
        self.df_max = df_max
        np.random.seed(random_state)
        self.startprob_ = None
        self.transmat_ = None
        self.means_ = None
        self.covars_ = None
        self.dfs_ = None
        self.converged_ = False
        self.n_iter_ = 0

    def _student_t_logpdf(
        self, X: np.ndarray, mean: np.ndarray, cov: np.ndarray, df: float
    ) -> np.ndarray:
        """Calculate Student-t log probability density."""
        n_samples, n_features = X.shape
        diff = X - mean
        cov_inv = np.linalg.inv(cov + 1e-06 * np.eye(n_features))
        mahal_dist = np.sum(diff @ cov_inv * diff, axis=1)
        log_prob = (
            gammaln((df + n_features) / 2)
            - gammaln(df / 2)
            - n_features / 2 * np.log(np.pi * df)
            - 0.5 * np.log(np.linalg.det(cov) + 1e-10)
            - (df + n_features) / 2 * np.log(1 + mahal_dist / df)
        )
        return log_prob

    def _forward(self, X: np.ndarray) -> Tuple[np.ndarray, float]:
        """Forward algorithm."""
        n_samples = len(X)
        log_alpha = np.zeros((n_samples, self.n_states))
        for j in range(self.n_states):
            log_alpha[0, j] = (
                np.log(self.startprob_[j] + 1e-10)
                + self._student_t_logpdf(X[0:1], self.means_[j], self.covars_[j], self.dfs_[j])[0]
            )
        for t in range(1, n_samples):
            for j in range(self.n_states):
                log_trans = np.log(self.transmat_[:, j] + 1e-10) + log_alpha[t - 1]
                log_alpha[t, j] = (
                    np.logaddexp.reduce(log_trans)
                    + self._student_t_logpdf(
                        X[t : t + 1], self.means_[j], self.covars_[j], self.dfs_[j]
                    )[0]
                )
        log_likelihood = np.logaddexp.reduce(log_alpha[-1])
        return (log_alpha, log_likelihood)

    def _backward(self, X: np.ndarray) -> np.ndarray:
        """Backward algorithm."""
        n_samples = len(X)
        log_beta = np.zeros((n_samples, self.n_states))
        for t in range(n_samples - 2, -1, -1):
            for i in range(self.n_states):
                log_emission = np.array(
                    [
                        self._student_t_logpdf(
                            X[t + 1 : t + 2], self.means_[j], self.covars_[j], self.dfs_[j]
                        )[0]
                        for j in range(self.n_states)
                    ]
                )
                log_beta[t, i] = np.logaddexp.reduce(
                    np.log(self.transmat_[i] + 1e-10) + log_emission + log_beta[t + 1]
                )
        return log_beta

    def _compute_posteriors(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Compute posterior probabilities."""
        n_samples = len(X)
        log_alpha, log_likelihood = self._forward(X)
        log_beta = self._backward(X)
        log_gamma = log_alpha + log_beta
        log_gamma -= np.logaddexp.reduce(log_gamma, axis=1, keepdims=True)
        gamma = np.exp(log_gamma)
        xi = np.zeros((n_samples - 1, self.n_states, self.n_states))
        for t in range(n_samples - 1):
            for i in range(self.n_states):
                for j in range(self.n_states):
                    log_emission = self._student_t_logpdf(
                        X[t + 1 : t + 2], self.means_[j], self.covars_[j], self.dfs_[j]
                    )[0]
                    xi[t, i, j] = np.exp(
                        log_alpha[t, i]
                        + np.log(self.transmat_[i, j] + 1e-10)
                        + log_emission
                        + log_beta[t + 1, j]
                        - log_likelihood
                    )
        return (gamma, xi)

    def fit(self, X: np.ndarray):
        """Fit Student-t HMM using EM algorithm."""
        n_samples, n_features = X.shape
        self.startprob_ = np.ones(self.n_states) / self.n_states
        self.transmat_ = np.ones((self.n_states, self.n_states)) / self.n_states
        from sklearn.cluster import KMeans

        kmeans = KMeans(n_clusters=self.n_states, random_state=self.random_state, n_init=10)
        labels = kmeans.fit_predict(X)
        self.means_ = np.array(
            [
                X[labels == i].mean(axis=0) if (labels == i).sum() > 0 else X.mean(axis=0)
                for i in range(self.n_states)
            ]
        )
        self.covars_ = np.array(
            [
                np.cov(X[labels == i].T) if (labels == i).sum() > 1 else np.eye(n_features)
                for i in range(self.n_states)
            ]
        )
        self.dfs_ = np.full(self.n_states, self.df_init)
        prev_log_likelihood = -np.inf
        for iteration in range(self.n_iter):
            gamma, xi = self._compute_posteriors(X)
            _, log_likelihood = self._forward(X)
            if abs(log_likelihood - prev_log_likelihood) < self.tol:
                self.converged_ = True
                self.n_iter_ = iteration + 1
                break
            prev_log_likelihood = log_likelihood
            self.startprob_ = gamma[0] + 1e-10
            self.startprob_ /= self.startprob_.sum()
            xi_sum = xi.sum(axis=0) + 1e-10
            self.transmat_ = xi_sum / xi_sum.sum(axis=1, keepdims=True)
            for j in range(self.n_states):
                gamma_sum = gamma[:, j].sum() + 1e-10
                self.means_[j] = gamma[:, j] @ X / gamma_sum
                diff = X - self.means_[j]
                self.covars_[j] = diff.T * gamma[:, j] @ diff / gamma_sum + 1e-06 * np.eye(
                    n_features
                )

                def neg_log_likelihood_df(df_val):
                    if df_val < self.df_min or df_val > self.df_max:
                        return 10000000000.0
                    log_prob = self._student_t_logpdf(X, self.means_[j], self.covars_[j], df_val)
                    return -(gamma[:, j] * log_prob).sum()

                from scipy.optimize import minimize_scalar

                result = minimize_scalar(
                    neg_log_likelihood_df, bounds=(self.df_min, self.df_max), method="bounded"
                )
                self.dfs_[j] = result.x
        if not self.converged_:
            self.n_iter_ = self.n_iter
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict state probabilities."""
        gamma, _ = self._compute_posteriors(X)
        return gamma

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict most likely state sequence using Viterbi."""
        n_samples = len(X)
        log_delta = np.zeros((n_samples, self.n_states))
        psi = np.zeros((n_samples, self.n_states), dtype=int)
        for j in range(self.n_states):
            log_delta[0, j] = (
                np.log(self.startprob_[j] + 1e-10)
                + self._student_t_logpdf(X[0:1], self.means_[j], self.covars_[j], self.dfs_[j])[0]
            )
        for t in range(1, n_samples):
            for j in range(self.n_states):
                trans_prob = log_delta[t - 1] + np.log(self.transmat_[:, j] + 1e-10)
                psi[t, j] = np.argmax(trans_prob)
                log_delta[t, j] = (
                    trans_prob[psi[t, j]]
                    + self._student_t_logpdf(
                        X[t : t + 1], self.means_[j], self.covars_[j], self.dfs_[j]
                    )[0]
                )
        states = np.zeros(n_samples, dtype=int)
        states[-1] = np.argmax(log_delta[-1])
        for t in range(n_samples - 2, -1, -1):
            states[t] = psi[t + 1, states[t + 1]]
        return states


class FeatureEngine:
    """
    Calculate features using adjusted close prices.

    CRITICAL FIX: All calculations use Adj_Close to handle splits/dividends correctly.
    """

    @staticmethod
    def calculate_features(df: pd.DataFrame, config: StrategyConfig, ticker: str) -> pd.DataFrame:
        """
        Calculate features using adjusted prices with T-1 lag.

        IMPROVEMENT: Uses Adj_Close for all calculations to avoid distortions.
        """
        df = df.copy()
        df["log_return"] = np.log(df["Adj_Close"] / df["Adj_Close"].shift(1))
        df["rolling_vol"] = (
            df["log_return"].rolling(window=config.VOL_WINDOW, min_periods=config.VOL_WINDOW).std()
        )
        momentum_scores = []
        for window, weight in zip(config.MOMENTUM_WINDOWS, config.MOMENTUM_WEIGHTS):
            momentum = df["Adj_Close"].pct_change(window)
            momentum_scores.append(momentum * weight)
        df["composite_momentum"] = sum(momentum_scores)
        df[f"{ticker}_return"] = df["log_return"]
        df[f"{ticker}_vol"] = df["rolling_vol"].shift(1)
        df[f"{ticker}_momentum"] = df["composite_momentum"].shift(1)
        return df

    @staticmethod
    def prepare_all_features(
        data_dict: Dict[str, pd.DataFrame], config: StrategyConfig
    ) -> Dict[str, pd.DataFrame]:
        """Prepare features for all assets."""
        print("\n" + "=" * 80)
        print("FEATURE ENGINEERING")
        print("=" * 80)
        features_dict = {}
        for ticker in config.MACRO_ASSETS + [config.VIX_TICKER]:
            data_ticker = ticker[1:] if ticker.startswith("^") else ticker
            if data_ticker not in data_dict:
                if ticker.startswith("^"):
                    data_ticker = ticker
                if data_ticker not in data_dict:
                    print(f"  âœ— {ticker}: Data not available")
                    continue
            df = data_dict[data_ticker].copy()
            df_features = FeatureEngine.calculate_features(df, config, ticker)
            features_dict[ticker] = df_features
            valid_features = df_features[
                [f"{ticker}_return", f"{ticker}_vol", f"{ticker}_momentum"]
            ].dropna()
            print(f"  âœ“ {ticker:6s}: {len(valid_features):4d} valid feature vectors")
        if len(features_dict) < 2:
            raise ValueError(f"Insufficient assets with features: {len(features_dict)}")
        print(f"\nâœ“ Features prepared for {len(features_dict)} assets")
        return features_dict


class MultiAssetConsensusDetector:
    """Multi-asset consensus regime detection using Student-t HMM."""

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.models = {}
        self.fold_history = []

    def walk_forward_predict(self, features_dict: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        """
        Walk-forward consensus regime detection.

        TIMING: Features calculated at T-1, predictions valid for T.
        """
        print("\n" + "=" * 80)
        print("WALK-FORWARD CONSENSUS REGIME DETECTION")
        print("=" * 80)
        date_sets = [df["Date"].values for df in features_dict.values()]
        common_dates = pd.to_datetime(sorted(set.intersection(*map(set, date_sets))))
        print(f"Common dates: {len(common_dates)}")
        print(f"Date range: {common_dates[0].date()} to {common_dates[-1].date()}")
        results = pd.DataFrame(
            {"Date": common_dates, "consensus": np.nan, "regime": np.nan, "regime_name": ""}
        )
        for ticker in features_dict.keys():
            results[f"{ticker}_state"] = np.nan
            results[f"{ticker}_proba"] = np.nan
        last_refit_idx = -self.config.REFIT_FREQUENCY
        current_models = {}
        for i in range(len(common_dates)):
            current_date = common_dates[i]
            if (
                i - last_refit_idx >= self.config.REFIT_FREQUENCY
                and i >= self.config.MIN_TRAINING_DAYS
            ):
                print(f"\nREFIT #{len(self.fold_history) + 1} at {current_date.date()}")
                print(
                    f"Training: {common_dates[0].date()} to {common_dates[i - 1].date()} ({i} days)"
                )
                fold_info = {
                    "refit_date": current_date,
                    "train_start": common_dates[0],
                    "train_end": common_dates[i - 1],
                    "train_size": i,
                    "models": {},
                }
                for ticker, df_features in features_dict.items():
                    print(f"  Training {ticker} HMM...")
                    train_dates = common_dates[:i]
                    train_df = df_features[df_features["Date"].isin(train_dates)].copy()
                    feature_cols = [f"{ticker}_return", f"{ticker}_vol", f"{ticker}_momentum"]
                    X_train = train_df[feature_cols].dropna().values
                    if len(X_train) < 100:
                        print(f"    âœ— Insufficient data: {len(X_train)} samples")
                        continue
                    scaler = StandardScaler()
                    X_train_scaled = scaler.fit_transform(X_train)
                    model = StudentTHMM(
                        n_states=self.config.HMM_N_STATES,
                        n_iter=self.config.HMM_N_ITER,
                        tol=self.config.HMM_CONVERGENCE_TOL,
                        random_state=self.config.HMM_RANDOM_STATE,
                        df_init=self.config.STUDENT_T_DF_INIT,
                        df_min=self.config.STUDENT_T_DF_MIN,
                        df_max=self.config.STUDENT_T_DF_MAX,
                    )
                    model.fit(X_train_scaled)
                    state_0_return = model.means_[0][0]
                    state_1_return = model.means_[1][0]
                    risk_on_state = 1 if state_1_return > state_0_return else 0
                    current_models[ticker] = {
                        "model": model,
                        "scaler": scaler,
                        "risk_on_state": risk_on_state,
                    }
                    fold_info["models"][ticker] = {
                        "converged": model.converged_,
                        "iterations": model.n_iter_,
                        "risk_on_state": risk_on_state,
                        "state_0_mean_return": state_0_return,
                        "state_1_mean_return": state_1_return,
                    }
                    print(f"    âœ“ Converged: {model.converged_}, Risk-on state: {risk_on_state}")
                self.fold_history.append(fold_info)
                last_refit_idx = i
            if i >= self.config.MIN_TRAINING_DAYS and current_models:
                predictions = {}
                for ticker, model_info in current_models.items():
                    df_features = features_dict[ticker]
                    current_row = df_features[df_features["Date"] == current_date]
                    if current_row.empty:
                        continue
                    feature_cols = [f"{ticker}_return", f"{ticker}_vol", f"{ticker}_momentum"]
                    X_current = current_row[feature_cols].values
                    if np.isnan(X_current).any():
                        continue
                    X_scaled = model_info["scaler"].transform(X_current)
                    state_proba = model_info["model"].predict_proba(X_scaled)[0]
                    state = model_info["model"].predict(X_scaled)[0]
                    risk_on_state = model_info["risk_on_state"]
                    risk_on_proba = state_proba[risk_on_state]
                    predictions[ticker] = risk_on_proba
                    results.loc[results["Date"] == current_date, f"{ticker}_state"] = state
                    results.loc[results["Date"] == current_date, f"{ticker}_proba"] = risk_on_proba
                if predictions:
                    total_weighted = 0
                    total_weights = 0
                for ticker in predictions:
                    weight = self.config.MACRO_WEIGHTS.get(ticker, 0)
                    if weight == 0 and ticker == "VIX":
                        weight = self.config.MACRO_WEIGHTS.get("^VIX", 0)
                    elif weight == 0 and ticker == "^VIX":
                        weight = self.config.MACRO_WEIGHTS.get("VIX", 0)
                    if ticker in ("^VIX", "VIX"):
                        total_weighted += (1 - predictions[ticker]) * weight
                    else:
                        total_weighted += predictions[ticker] * weight
                    total_weights += weight
                    consensus = total_weighted / total_weights if total_weights > 0 else 0.5
                    results.loc[results["Date"] == current_date, "consensus"] = consensus
                    if consensus > self.config.STRONG_RISK_ON_THRESHOLD:
                        regime = 4
                        regime_name = "STRONG_RISK_ON"
                    elif consensus > self.config.MODERATE_RISK_ON_THRESHOLD:
                        regime = 3
                        regime_name = "MODERATE_RISK_ON"
                    elif consensus > self.config.MODERATE_RISK_OFF_THRESHOLD:
                        regime = 2
                        regime_name = "NEUTRAL"
                    elif consensus > self.config.STRONG_RISK_OFF_THRESHOLD:
                        regime = 1
                        regime_name = "MODERATE_RISK_OFF"
                    else:
                        regime = 0
                        regime_name = "STRONG_RISK_OFF"
                    results.loc[results["Date"] == current_date, "regime"] = regime
                    results.loc[results["Date"] == current_date, "regime_name"] = regime_name
        print(f"\n{'=' * 80}")
        print("REGIME DETECTION SUMMARY")
        print(f"{'=' * 80}")
        print(f"Total folds: {len(self.fold_history)}")
        print(f"Predictions: {results['regime'].notna().sum()}")
        regime_counts = results["regime_name"].value_counts()
        print(f"\nRegime distribution:")
        for regime_name in [
            "STRONG_RISK_ON",
            "MODERATE_RISK_ON",
            "NEUTRAL",
            "MODERATE_RISK_OFF",
            "STRONG_RISK_OFF",
        ]:
            count = regime_counts.get(regime_name, 0)
            pct = count / len(results) * 100 if len(results) > 0 else 0
            print(f"  {regime_name:20s}: {count:4d} days ({pct:5.1f}%)")
        return results


class MaxSharpeOptimizer:
    """Maximum Sharpe Ratio optimizer with leverage support."""

    def __init__(self, risk_free_rate: float, target_exposure: float):
        self.risk_free_rate = risk_free_rate
        self.target_exposure = target_exposure

    def optimize(
        self, returns: pd.DataFrame, min_weight: float, max_weight: float
    ) -> Dict[str, float]:
        """Optimize for maximum Sharpe ratio with adjusted close returns."""
        if len(returns) < 60:
            return {}
        expected_returns = returns.mean() * 252
        cov_matrix = returns.cov() * 252
        try:
            lw = LedoitWolf()
            cov_shrunk, shrinkage = (lw.fit(returns.values).covariance_, lw.shrinkage_)
            cov_matrix = pd.DataFrame(
                cov_shrunk * 252, index=returns.columns, columns=returns.columns
            )
        except:
            pass
        n_assets = len(expected_returns)

        def neg_sharpe(weights):
            port_return = np.dot(weights, expected_returns)
            port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
            if port_vol == 0:
                return 10000000000.0
            return -(port_return - self.risk_free_rate) / port_vol

        constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - self.target_exposure}]
        bounds = [(0, max_weight) for _ in range(n_assets)]
        x0 = np.array([self.target_exposure / n_assets] * n_assets)
        try:
            result = minimize(
                neg_sharpe,
                x0,
                method="SLSQP",
                bounds=bounds,
                constraints=constraints,
                options={"maxiter": 1000, "ftol": 1e-09},
            )
            if result.success:
                weights = result.x
                weights = np.where(weights < min_weight, 0, weights)
                if np.sum(weights) > 0:
                    weights = weights * self.target_exposure / np.sum(weights)
                    return {
                        asset: w
                        for asset, w in zip(expected_returns.index, weights)
                        if w >= min_weight
                    }
        except:
            pass
        return {}


class HRPOptimizer:
    """Hierarchical Risk Parity optimizer."""

    def __init__(self, target_exposure: float):
        self.target_exposure = target_exposure

    def _get_cluster_var(self, cov: np.ndarray, cluster_items: list) -> float:
        """Calculate cluster variance."""
        cluster_cov = cov[np.ix_(cluster_items, cluster_items)]
        weights = np.ones(len(cluster_items)) / len(cluster_items)
        return np.dot(weights, np.dot(cluster_cov, weights))

    def _recursive_bisection(self, cov: np.ndarray, sort_idx: list) -> np.ndarray:
        """Recursive bisection for HRP allocation."""
        weights = np.ones(len(sort_idx))
        clusters = [sort_idx]
        while len(clusters) > 0:
            clusters = [
                cluster[start:end]
                for cluster in clusters
                for start, end in [(0, len(cluster) // 2), (len(cluster) // 2, len(cluster))]
                if len(cluster) > 1
            ]
            for i in range(0, len(clusters), 2):
                if i + 1 < len(clusters):
                    cluster_0 = clusters[i]
                    cluster_1 = clusters[i + 1]
                    var_0 = self._get_cluster_var(cov, cluster_0)
                    var_1 = self._get_cluster_var(cov, cluster_1)
                    alpha = 1 - var_0 / (var_0 + var_1)
                    weights[cluster_0] *= alpha
                    weights[cluster_1] *= 1 - alpha
        return weights

    def optimize(self, returns: pd.DataFrame) -> Dict[str, float]:
        """Optimize using HRP with adjusted close returns."""
        if len(returns) < 60:
            return {}
        corr_matrix = returns.corr()
        dist_matrix = np.sqrt((1 - corr_matrix) / 2)
        dist_condensed = squareform(dist_matrix.values, checks=False)
        link = linkage(dist_condensed, method="single")
        sort_idx = self._get_quasi_diag(link, len(corr_matrix))
        cov_matrix = returns.cov().values
        hrp_weights = self._recursive_bisection(cov_matrix, sort_idx)
        hrp_weights = hrp_weights * self.target_exposure / hrp_weights.sum()
        return {asset: w for asset, w in zip(returns.columns, hrp_weights) if w > 0.01}

    def _get_quasi_diag(self, link: np.ndarray, n: int) -> list:
        """Get quasi-diagonal order from linkage matrix."""
        sort_idx = []

        def recurse(node):
            if node < n:
                sort_idx.append(node)
            else:
                left = int(link[node - n, 0])
                right = int(link[node - n, 1])
                recurse(left)
                recurse(right)

        recurse(2 * n - 2)
        return sort_idx


class AssetScorer:
    """Score assets using multi-timeframe momentum with adjusted prices."""

    @staticmethod
    def calculate_scores(
        data_dict: Dict[str, pd.DataFrame],
        date: pd.Timestamp,
        asset_universe: List[str],
        config: StrategyConfig,
    ) -> pd.Series:
        """
        Calculate momentum scores using adjusted close prices.

        TIMING: Uses data up to date (T-1) for scoring at Close[T-1].
        """
        scores = {}
        for ticker in asset_universe:
            if ticker not in data_dict:
                continue
            df = data_dict[ticker]
            df_up_to_date = df[df["Date"] <= date].copy()
            if len(df_up_to_date) < max(config.MOMENTUM_WINDOWS) + config.VOL_WINDOW:
                continue
            momentum_score = 0
            for window, weight in zip(config.MOMENTUM_WINDOWS, config.MOMENTUM_WEIGHTS):
                momentum = (
                    df_up_to_date["Adj_Close"].iloc[-1]
                    / df_up_to_date["Adj_Close"].iloc[-window - 1]
                    - 1
                )
                momentum_score += momentum * weight
            returns = df_up_to_date["Adj_Close"].pct_change().iloc[-(config.VOL_WINDOW + 1) :]
            volatility = returns.std()
            if volatility > 0:
                scores[ticker] = momentum_score / volatility
            else:
                scores[ticker] = 0
        return pd.Series(scores).sort_values(ascending=False)


class RiskManager:
    """CVaR-based stop-loss system with proper timing."""

    @staticmethod
    def calculate_cvar_stop(df: pd.DataFrame, date: pd.Timestamp, config: StrategyConfig) -> float:
        """
        Calculate CVaR stop-loss using adjusted close returns.

        TIMING: Uses historical data up to date (T-1).
        """
        df_history = df[df["Date"] <= date].copy()
        if len(df_history) < config.CVAR_LOOKBACK:
            return -0.15
        returns = df_history["Adj_Close"].pct_change().iloc[-config.CVAR_LOOKBACK :]
        var_threshold = returns.quantile(1 - config.CVAR_CONFIDENCE)
        losses_beyond_var = returns[returns < var_threshold]
        if len(losses_beyond_var) > 0:
            cvar = losses_beyond_var.mean()
        else:
            cvar = var_threshold
        stop_loss = cvar * config.STOP_LOSS_BUFFER
        return max(stop_loss, -0.25)

    @staticmethod
    def check_stop_loss(entry_price: float, current_price: float, stop_level: float) -> bool:
        """Check if position breached stop-loss."""
        current_return = (current_price - entry_price) / entry_price
        return current_return <= stop_level


class BacktestEngine:
    """
    Production backtest engine with proper timing and leverage accounting.

    EXECUTION FLOW (FIXED):
    Day T-1 (Signal Generation):
      â€¢ Portfolio valued at Close[T-1]
      â€¢ Features calculated from T-1 data
      â€¢ Regime detected
      â€¢ Signals generated
      â€¢ Stop-losses checked at Close[T-1]

    Day T (Execution):
      â€¢ Execute signals at Open[T]
      â€¢ Stop-loss exits at Open[T] with slippage
      â€¢ Portfolio valued at Open[T]

    Day T to T+1 (Holding):
      â€¢ Portfolio valued at Close[T]
      â€¢ Returns: Open[T+1]/Open[T] - 1 (on execution days)
      â€¢ Daily financing costs accrued if leveraged

    LEVERAGE MECHANICS:
    â€¢ Negative cash allowed up to MAX_LEVERAGE
    â€¢ Daily interest: max(0, gross_exposure - equity) Ã— rate/252
    â€¢ Hard cap enforcement at rebalance
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.trades = []
        self.stop_loss_triggers = []
        self.position_tracker = {}
        self.daily_financing_costs = []

    def calculate_transaction_costs(self, trade_value: float) -> float:
        """Calculate realistic transaction costs."""
        abs_value = abs(trade_value)
        commission = max(abs_value * self.config.COMMISSION_RATE, self.config.MIN_COMMISSION)
        spread = abs_value * self.config.BID_ASK_SPREAD
        impact = abs_value * self.config.MARKET_IMPACT
        return commission + spread + impact

    def calculate_financing_cost(self, gross_exposure: float, equity: float) -> float:
        """
        Calculate daily financing cost for borrowed capital.

        FORMULA: interest = max(0, gross_exposure - equity) Ã— annual_rate / 252

        This represents the cost of borrowing to maintain leveraged positions.
        """
        borrowed = max(0, gross_exposure - equity)
        daily_rate = self.config.MARGIN_RATE_ANNUAL / 252
        return borrowed * daily_rate

    def select_assets_by_regime(self, regime: int) -> List[str]:
        """Select asset universe based on regime."""
        if regime == 4:
            return self.config.HIGH_BETA_ASSETS
        elif regime == 3:
            return self.config.BALANCED_ASSETS
        elif regime == 2:
            return self.config.NEUTRAL_ASSETS
        elif regime == 1:
            return self.config.DEFENSIVE_ASSETS
        else:
            return self.config.SAFE_HAVEN_ASSETS

    def get_regime_parameters(self, regime: int) -> dict:
        """Get regime-specific parameters."""
        params = {
            4: {
                "exposure": self.config.STRONG_RISK_ON_EXPOSURE,
                "max_positions": self.config.STRONG_RISK_ON_MAX_POSITIONS,
                "min_weight": self.config.STRONG_RISK_ON_MIN_WEIGHT,
                "max_weight": self.config.STRONG_RISK_ON_MAX_WEIGHT,
                "optimization": "MaxSharpe",
                "hedge": False,
            },
            3: {
                "exposure": self.config.MODERATE_RISK_ON_EXPOSURE,
                "max_positions": self.config.MODERATE_RISK_ON_MAX_POSITIONS,
                "min_weight": self.config.MODERATE_RISK_ON_MIN_WEIGHT,
                "max_weight": self.config.MODERATE_RISK_ON_MAX_WEIGHT,
                "optimization": "MaxSharpe",
                "hedge": False,
            },
            2: {
                "exposure": self.config.NEUTRAL_EXPOSURE,
                "max_positions": self.config.NEUTRAL_MAX_POSITIONS,
                "min_weight": self.config.NEUTRAL_MIN_WEIGHT,
                "max_weight": self.config.NEUTRAL_MAX_WEIGHT,
                "optimization": "HRP",
                "hedge": False,
            },
            1: {
                "exposure": self.config.MODERATE_RISK_OFF_EXPOSURE,
                "max_positions": self.config.MODERATE_RISK_OFF_MAX_POSITIONS,
                "min_weight": self.config.DEFENSIVE_MIN_WEIGHT,
                "max_weight": self.config.DEFENSIVE_MAX_WEIGHT,
                "optimization": "HRP",
                "hedge": False,
            },
            0: {
                "exposure": self.config.STRONG_RISK_OFF_LONG_EXPOSURE,
                "max_positions": self.config.STRONG_RISK_OFF_MAX_POSITIONS,
                "min_weight": self.config.DEFENSIVE_MIN_WEIGHT,
                "max_weight": self.config.DEFENSIVE_MAX_WEIGHT,
                "optimization": "HRP",
                "hedge": True,
                "hedge_exposure": self.config.STRONG_RISK_OFF_HEDGE_EXPOSURE,
            },
        }
        return params[regime]

    def run_backtest(
        self, regime_predictions: pd.DataFrame, data_dict: Dict[str, pd.DataFrame]
    ) -> Dict:
        """
        Execute walk-forward backtest with proper timing and leverage.

        CRITICAL FIXES:
        1. Close[T-1] signal â†’ Open[T] execution
        2. Open-to-Open returns on execution days
        3. True leverage with financing costs
        4. Benchmark aligned to same timing
        5. Stop-losses exit at Open[T+1] with slippage
        """
        print("\n" + "=" * 80)
        print("BACKTEST EXECUTION (FIXED TIMING)")
        print("=" * 80)
        backtest_start = pd.to_datetime(self.config.BACKTEST_START_DATE)
        backtest_end = pd.to_datetime(self.config.BACKTEST_END_DATE)
        backtest_regime = (
            regime_predictions[
                (regime_predictions["Date"] >= backtest_start)
                & (regime_predictions["Date"] <= backtest_end)
                & regime_predictions["regime"].notna()
            ]
            .copy()
            .reset_index(drop=True)
        )
        if len(backtest_regime) == 0:
            raise ValueError("No valid regime predictions")
        print(
            f"Period: {backtest_regime['Date'].min().date()} to {backtest_regime['Date'].max().date()}"
        )
        print(f"Trading days: {len(backtest_regime)}")
        print(f"\nExecution: Close[T-1] signal â†’ Open[T] execution")
        print(f"Returns: Open[T+1]/Open[T] - 1")
        print(
            f"Leverage: Max {self.config.MAX_LEVERAGE:.1%}, Rate: {self.config.MARGIN_RATE_ANNUAL:.2%}/year"
        )
        portfolio_values = []
        daily_returns = []
        positions_held = []
        regime_history = []
        consensus_history = []
        exposure_history = []
        cash_history = []
        leverage_history = []
        financing_history = []
        equity = self.config.INITIAL_CAPITAL
        cash = self.config.INITIAL_CAPITAL
        current_positions = {}
        stop_loss_count = 0
        regime_days = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
        total_financing_costs = 0
        last_rebalance = -self.config.REBALANCE_FREQUENCY
        bench_shares = 0
        if self.config.BENCHMARK_TICKER in data_dict:
            bench_df = data_dict[self.config.BENCHMARK_TICKER]
            first_date = backtest_regime["Date"].iloc[0]
            first_idx = bench_df[bench_df["Date"] == first_date].index
            if len(first_idx) > 0 and first_idx[0] + 1 < len(bench_df):
                bench_initial_price = bench_df.iloc[first_idx[0] + 1]["Adj_Open"]
                bench_shares = self.config.INITIAL_CAPITAL / bench_initial_price
                print(
                    f"\nBenchmark ({self.config.BENCHMARK_TICKER}): Open[0] = ${bench_initial_price:.2f}, Shares = {bench_shares:.2f}"
                )
        for i in range(len(backtest_regime)):
            current_date = backtest_regime.loc[i, "Date"]
            current_regime = int(backtest_regime.loc[i, "regime"])
            current_consensus = backtest_regime.loc[i, "consensus"]
            regime_days[current_regime] += 1
            if i + 1 >= len(backtest_regime):
                break
            next_date = backtest_regime.loc[i + 1, "Date"]
            if self.config.STOP_LOSS_ENABLED and current_positions:
                positions_to_close = []
                for ticker, pos_info in current_positions.items():
                    if ticker not in data_dict:
                        continue
                    df_ticker = data_dict[ticker]
                    price_row = df_ticker[df_ticker["Date"] == current_date]
                    if price_row.empty:
                        continue
                    current_price = price_row.iloc[0]["Adj_Close"]
                    entry_price = pos_info["entry_price"]
                    stop_level = RiskManager.calculate_cvar_stop(
                        df_ticker, current_date, self.config
                    )
                    if RiskManager.check_stop_loss(entry_price, current_price, stop_level):
                        positions_to_close.append(ticker)
                        stop_loss_count += 1
                        self.stop_loss_triggers.append(
                            {
                                "signal_date": current_date,
                                "execution_date": next_date,
                                "ticker": ticker,
                                "entry_price": entry_price,
                                "signal_price": current_price,
                                "return": (current_price - entry_price) / entry_price,
                                "stop_level": stop_level,
                            }
                        )
                for ticker in positions_to_close:
                    pos_info = current_positions[ticker]
                    df_ticker = data_dict[ticker]
                    next_price_row = df_ticker[df_ticker["Date"] == next_date]
                    if not next_price_row.empty:
                        exit_price = next_price_row.iloc[0]["Adj_Open"] * (
                            1 - self.config.STOP_LOSS_SLIPPAGE
                        )
                        proceeds = pos_info["shares"] * exit_price
                        costs = self.calculate_transaction_costs(proceeds)
                        cash += proceeds - costs
                        self.trades.append(
                            {
                                "signal_date": current_date,
                                "execution_date": next_date,
                                "ticker": ticker,
                                "action": "STOP_LOSS",
                                "shares": pos_info["shares"],
                                "price": exit_price,
                                "value": proceeds,
                                "costs": costs,
                                "regime": current_regime,
                            }
                        )
                        del current_positions[ticker]
            should_rebalance = i - last_rebalance >= self.config.REBALANCE_FREQUENCY
            if should_rebalance and i > 0:
                regime_params = self.get_regime_parameters(current_regime)
                asset_universe = self.select_assets_by_regime(current_regime)
                scores = AssetScorer.calculate_scores(
                    data_dict, current_date, asset_universe, self.config
                )
                top_assets = scores.head(regime_params["max_positions"])
                if len(top_assets) > 0:
                    returns_list = []
                    valid_tickers = []
                    for ticker in top_assets.index:
                        if ticker not in data_dict:
                            continue
                        df_ticker = data_dict[ticker]
                        df_history = df_ticker[df_ticker["Date"] <= current_date].copy()
                        if len(df_history) < 252:
                            continue
                        returns = df_history["Adj_Close"].pct_change().iloc[-252:]
                        returns_list.append(returns.values)
                        valid_tickers.append(ticker)
                    if len(valid_tickers) >= 2:
                        returns_df = pd.DataFrame(
                            np.column_stack(returns_list), columns=valid_tickers
                        ).dropna()
                        if regime_params["optimization"] == "MaxSharpe":
                            optimizer = MaxSharpeOptimizer(
                                self.config.RISK_FREE_RATE, regime_params["exposure"]
                            )
                            target_weights = optimizer.optimize(
                                returns_df, regime_params["min_weight"], regime_params["max_weight"]
                            )
                        else:
                            optimizer = HRPOptimizer(regime_params["exposure"])
                            target_weights = optimizer.optimize(returns_df)
                            if target_weights:
                                for ticker in list(target_weights.keys()):
                                    if target_weights[ticker] < regime_params["min_weight"]:
                                        del target_weights[ticker]
                                    elif target_weights[ticker] > regime_params["max_weight"]:
                                        target_weights[ticker] = regime_params["max_weight"]
                                total = sum(target_weights.values())
                                if total > 0:
                                    target_weights = {
                                        t: w * regime_params["exposure"] / total
                                        for t, w in target_weights.items()
                                    }
                        if regime_params.get("hedge", False):
                            target_weights[self.config.HEDGE_TICKER] = regime_params[
                                "hedge_exposure"
                            ]
                        if target_weights:
                            current_equity = cash
                            for ticker, pos_info in current_positions.items():
                                if ticker not in data_dict:
                                    continue
                                df_ticker = data_dict[ticker]
                                price_row = df_ticker[df_ticker["Date"] == current_date]
                                if not price_row.empty:
                                    current_price = price_row.iloc[0]["Adj_Close"]
                                    current_equity += pos_info["shares"] * current_price
                            target_gross_exposure = sum(target_weights.values()) * current_equity
                            if target_gross_exposure > current_equity * self.config.MAX_LEVERAGE:
                                scale_factor = (
                                    current_equity
                                    * self.config.MAX_LEVERAGE
                                    / target_gross_exposure
                                )
                                target_weights = {
                                    k: v * scale_factor for k, v in target_weights.items()
                                }
                                print(f"  ! Leverage capped: Scaled weights by {scale_factor:.3f}")
                            for ticker in list(current_positions.keys()):
                                if ticker not in target_weights:
                                    pos_info = current_positions[ticker]
                                    df_ticker = data_dict[ticker]
                                    next_price_row = df_ticker[df_ticker["Date"] == next_date]
                                    if not next_price_row.empty:
                                        exit_price = next_price_row.iloc[0]["Adj_Open"]
                                        proceeds = pos_info["shares"] * exit_price
                                        costs = self.calculate_transaction_costs(proceeds)
                                        cash += proceeds - costs
                                        self.trades.append(
                                            {
                                                "signal_date": current_date,
                                                "execution_date": next_date,
                                                "ticker": ticker,
                                                "action": "SELL",
                                                "shares": pos_info["shares"],
                                                "price": exit_price,
                                                "value": proceeds,
                                                "costs": costs,
                                                "regime": current_regime,
                                            }
                                        )
                                        del current_positions[ticker]
                            for ticker, target_weight in target_weights.items():
                                if ticker not in data_dict:
                                    continue
                                df_ticker = data_dict[ticker]
                                next_price_row = df_ticker[df_ticker["Date"] == next_date]
                                if next_price_row.empty:
                                    continue
                                entry_price = next_price_row.iloc[0]["Adj_Open"]
                                target_value = current_equity * target_weight
                                target_shares = target_value / entry_price
                                current_shares = current_positions.get(ticker, {}).get("shares", 0)
                                shares_diff = target_shares - current_shares
                                if (
                                    abs(shares_diff * entry_price / current_equity)
                                    > self.config.MIN_POSITION_CHANGE
                                ):
                                    trade_value = shares_diff * entry_price
                                    costs = self.calculate_transaction_costs(trade_value)
                                    if shares_diff > 0:
                                        cash -= trade_value + costs
                                        current_positions[ticker] = {
                                            "shares": target_shares,
                                            "entry_price": entry_price,
                                            "weight": target_weight,
                                        }
                                        self.trades.append(
                                            {
                                                "signal_date": current_date,
                                                "execution_date": next_date,
                                                "ticker": ticker,
                                                "action": "BUY",
                                                "shares": shares_diff,
                                                "price": entry_price,
                                                "value": trade_value,
                                                "costs": costs,
                                                "regime": current_regime,
                                            }
                                        )
                                    else:
                                        cash += -trade_value - costs
                                        if target_shares > 0:
                                            current_positions[ticker] = {
                                                "shares": target_shares,
                                                "entry_price": entry_price,
                                                "weight": target_weight,
                                            }
                                        elif ticker in current_positions:
                                            del current_positions[ticker]
                                        self.trades.append(
                                            {
                                                "signal_date": current_date,
                                                "execution_date": next_date,
                                                "ticker": ticker,
                                                "action": "SELL",
                                                "shares": -shares_diff,
                                                "price": entry_price,
                                                "value": -trade_value,
                                                "costs": costs,
                                                "regime": current_regime,
                                            }
                                        )
                            last_rebalance = i
            gross_exposure = 0
            equity = cash
            for ticker, pos_info in current_positions.items():
                if ticker not in data_dict:
                    continue
                df_ticker = data_dict[ticker]
                price_row = df_ticker[df_ticker["Date"] == current_date]
                if not price_row.empty:
                    current_price = price_row.iloc[0]["Adj_Close"]
                    position_value = pos_info["shares"] * current_price
                    equity += position_value
                    gross_exposure += abs(position_value)
            financing_cost = self.calculate_financing_cost(gross_exposure, equity)
            equity -= financing_cost
            total_financing_costs += financing_cost
            portfolio_values.append(equity)
            positions_held.append(len(current_positions))
            regime_history.append(current_regime)
            consensus_history.append(current_consensus)
            leverage = gross_exposure / equity if equity > 0 else 1.0
            leverage_history.append(leverage)
            financing_history.append(financing_cost)
            exposure_history.append(gross_exposure / self.config.INITIAL_CAPITAL)
            cash_history.append(cash)
            if i > 0:
                daily_return = (equity - portfolio_values[i - 1]) / portfolio_values[i - 1]
            else:
                daily_return = 0
            daily_returns.append(daily_return)
        results_df = pd.DataFrame(
            {
                "Date": backtest_regime["Date"][: len(portfolio_values)],
                "Portfolio_Value": portfolio_values,
                "Daily_Return": daily_returns,
                "Positions": positions_held,
                "Regime": regime_history,
                "Consensus": consensus_history,
                "Exposure": exposure_history,
                "Cash": cash_history,
                "Leverage": leverage_history,
                "Financing_Cost": financing_history,
            }
        )
        results_df["Cumulative_Return"] = (1 + results_df["Daily_Return"]).cumprod()
        if bench_shares > 0:
            bench_values = []
            bench_returns = []
            bench_df = data_dict[self.config.BENCHMARK_TICKER]
            for i, date in enumerate(results_df["Date"]):
                bench_row = bench_df[bench_df["Date"] == date]
                if not bench_row.empty:
                    bench_price = bench_row.iloc[0]["Adj_Close"]
                    bench_value = bench_shares * bench_price
                    bench_values.append(bench_value)
                    if i > 0:
                        bench_ret = (bench_value - bench_values[i - 1]) / bench_values[i - 1]
                    else:
                        bench_ret = 0
                    bench_returns.append(bench_ret)
                else:
                    bench_values.append(
                        bench_values[-1] if bench_values else self.config.INITIAL_CAPITAL
                    )
                    bench_returns.append(0)
            results_df["Benchmark_Value"] = bench_values
            results_df["Benchmark_Return"] = bench_returns
            results_df["Benchmark_Cumulative"] = (1 + results_df["Benchmark_Return"]).cumprod()
        total_return = (
            portfolio_values[-1] - self.config.INITIAL_CAPITAL
        ) / self.config.INITIAL_CAPITAL
        annual_return = (1 + total_return) ** (252 / len(results_df)) - 1
        volatility = results_df["Daily_Return"].std() * np.sqrt(252)
        sharpe_ratio = (
            (annual_return - self.config.RISK_FREE_RATE) / volatility if volatility > 0 else 0
        )
        cumulative = results_df["Cumulative_Return"]
        running_max = cumulative.expanding().max()
        drawdowns = (cumulative - running_max) / running_max
        max_drawdown = drawdowns.min()
        if bench_shares > 0:
            bench_total_return = (
                bench_values[-1] - self.config.INITIAL_CAPITAL
            ) / self.config.INITIAL_CAPITAL
            bench_annual_return = (1 + bench_total_return) ** (252 / len(results_df)) - 1
            bench_volatility = results_df["Benchmark_Return"].std() * np.sqrt(252)
            bench_sharpe = (
                (bench_annual_return - self.config.RISK_FREE_RATE) / bench_volatility
                if bench_volatility > 0
                else 0
            )
            bench_cumulative = results_df["Benchmark_Cumulative"]
            bench_running_max = bench_cumulative.expanding().max()
            bench_drawdowns = (bench_cumulative - bench_running_max) / bench_running_max
            bench_max_dd = bench_drawdowns.min()
        else:
            bench_annual_return = 0
            bench_volatility = 0
            bench_sharpe = 0
            bench_max_dd = 0
        results = {
            "results_df": results_df,
            "trades": self.trades,
            "stop_loss_triggers": self.stop_loss_triggers,
            "total_return": total_return,
            "annual_return": annual_return,
            "volatility": volatility,
            "sharpe_ratio": sharpe_ratio,
            "max_drawdown": max_drawdown,
            "stop_loss_count": stop_loss_count,
            "total_trades": len(self.trades),
            "regime_days": regime_days,
            "avg_positions": results_df["Positions"].mean(),
            "avg_exposure": results_df["Exposure"].mean(),
            "avg_leverage": results_df["Leverage"].mean(),
            "max_leverage": results_df["Leverage"].max(),
            "total_financing_costs": total_financing_costs,
            "benchmark_annual_return": bench_annual_return,
            "benchmark_volatility": bench_volatility,
            "benchmark_sharpe": bench_sharpe,
            "benchmark_max_drawdown": bench_max_dd,
        }
        print(f"\n{'=' * 80}")
        print("BACKTEST SUMMARY")
        print(f"{'=' * 80}")
        print(f"Total Return: {total_return:.2%}")
        print(f"Annual Return: {annual_return:.2%}")
        print(f"Volatility: {volatility:.2%}")
        print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
        print(f"Max Drawdown: {max_drawdown:.2%}")
        print(f"\nLeverage:")
        print(f"Average: {results_df['Leverage'].mean():.2f}x")
        print(f"Maximum: {results_df['Leverage'].max():.2f}x")
        print(f"Total Financing Costs: ${total_financing_costs:,.2f}")
        print(f"\nBenchmark ({self.config.BENCHMARK_TICKER}):")
        print(f"Annual Return: {bench_annual_return:.2%}")
        print(f"Sharpe Ratio: {bench_sharpe:.2f}")
        print(f"Max Drawdown: {bench_max_dd:.2%}")
        print(f"\nTrading:")
        print(f"Total Trades: {len(self.trades)}")
        print(f"Stop Losses: {stop_loss_count}")
        print(f"Avg Positions: {results_df['Positions'].mean():.1f}")
        return results


class Visualizer:
    """Comprehensive visualization suite."""

    @staticmethod
    def create_dashboard(results: dict, config: StrategyConfig):
        """Create comprehensive dashboard."""
        fig = plt.figure(figsize=(20, 18))
        gs = fig.add_gridspec(7, 2, hspace=0.3, wspace=0.3)
        results_df = results["results_df"]
        regime_colors = {0: "#8B0000", 1: "#FF6B6B", 2: "#808080", 3: "#90EE90", 4: "#006400"}
        regime_names = {
            0: "STRONG RISK-OFF",
            1: "MODERATE RISK-OFF",
            2: "NEUTRAL",
            3: "MODERATE RISK-ON",
            4: "STRONG RISK-ON",
        }
        ax1 = fig.add_subplot(gs[0, :])
        for regime_idx in range(5):
            mask = results_df["Regime"] == regime_idx
            if mask.any():
                for date in results_df.loc[mask, "Date"]:
                    ax1.axvspan(
                        date,
                        date + pd.Timedelta(days=1),
                        color=regime_colors[regime_idx],
                        alpha=0.2,
                    )
        ax1.plot(
            results_df["Date"],
            results_df["Consensus"],
            linewidth=2,
            color="darkblue",
            label="Consensus",
        )
        ax1.axhline(config.STRONG_RISK_ON_THRESHOLD, color="darkgreen", linestyle="--", alpha=0.5)
        ax1.axhline(config.MODERATE_RISK_ON_THRESHOLD, color="green", linestyle="--", alpha=0.5)
        ax1.axhline(config.MODERATE_RISK_OFF_THRESHOLD, color="orange", linestyle="--", alpha=0.5)
        ax1.axhline(config.STRONG_RISK_OFF_THRESHOLD, color="red", linestyle="--", alpha=0.5)
        ax1.set_ylabel("Consensus", fontsize=11, fontweight="bold")
        ax1.set_title("Consensus and Regime Timeline", fontsize=13, fontweight="bold")
        ax1.grid(alpha=0.3)
        ax1.set_ylim(0, 1)
        ax2 = fig.add_subplot(gs[1, :])
        ax2.plot(
            results_df["Date"],
            results_df["Leverage"],
            linewidth=1.5,
            color="purple",
            label="Leverage",
        )
        ax2.axhline(
            config.MAX_LEVERAGE,
            color="red",
            linestyle="--",
            label=f"Max Leverage ({config.MAX_LEVERAGE:.1%})",
        )
        ax2.fill_between(
            results_df["Date"],
            1,
            results_df["Leverage"],
            where=results_df["Leverage"] > 1,
            alpha=0.3,
            color="purple",
        )
        ax2.set_ylabel("Leverage (x)", fontsize=11, fontweight="bold")
        ax2.set_title("Portfolio Leverage Over Time", fontsize=13, fontweight="bold")
        ax2.legend(fontsize=9)
        ax2.grid(alpha=0.3)
        ax3 = fig.add_subplot(gs[2, :])
        cumulative_financing = results_df["Financing_Cost"].cumsum()
        ax3.plot(
            results_df["Date"],
            cumulative_financing,
            linewidth=2,
            color="red",
            label="Cumulative Financing Costs",
        )
        ax3.fill_between(results_df["Date"], 0, cumulative_financing, alpha=0.3, color="red")
        ax3.set_ylabel("Financing Cost ($)", fontsize=11, fontweight="bold")
        ax3.set_title("Cumulative Financing Costs from Leverage", fontsize=13, fontweight="bold")
        ax3.legend(fontsize=9)
        ax3.grid(alpha=0.3)
        ax4 = fig.add_subplot(gs[3, :])
        ax4.bar(results_df["Date"], results_df["Positions"], width=1, color="coral", alpha=0.7)
        ax4.set_ylabel("Positions", fontsize=11, fontweight="bold")
        ax4.set_title("Position Count", fontsize=13, fontweight="bold")
        ax4.grid(alpha=0.3)
        ax5 = fig.add_subplot(gs[4, :])
        ax5.plot(
            results_df["Date"],
            results_df["Cumulative_Return"] * 100,
            linewidth=2.5,
            color="darkblue",
            label="FAMRS",
        )
        if "Benchmark_Cumulative" in results_df.columns:
            ax5.plot(
                results_df["Date"],
                results_df["Benchmark_Cumulative"] * 100,
                linewidth=2,
                color="orange",
                alpha=0.7,
                label=f"{config.BENCHMARK_TICKER}",
            )
        ax5.set_ylabel("Cumulative Return (Base 100)", fontsize=11, fontweight="bold")
        ax5.set_title("Strategy vs Benchmark", fontsize=13, fontweight="bold")
        ax5.legend(fontsize=10)
        ax5.grid(alpha=0.3)
        ax6 = fig.add_subplot(gs[5, :])
        cumulative = results_df["Cumulative_Return"]
        running_max = cumulative.expanding().max()
        drawdowns = (cumulative - running_max) / running_max * 100
        ax6.fill_between(results_df["Date"], drawdowns, 0, alpha=0.5, color="red")
        ax6.plot(results_df["Date"], drawdowns, linewidth=1.5, color="darkred")
        if "Benchmark_Cumulative" in results_df.columns:
            bench_cum = results_df["Benchmark_Cumulative"]
            bench_max = bench_cum.expanding().max()
            bench_dd = (bench_cum - bench_max) / bench_max * 100
            ax6.plot(results_df["Date"], bench_dd, linewidth=1.5, color="orange", alpha=0.7)
        ax6.set_ylabel("Drawdown (%)", fontsize=11, fontweight="bold")
        ax6.set_xlabel("Date", fontsize=11, fontweight="bold")
        ax6.set_title("Drawdown Analysis", fontsize=13, fontweight="bold")
        ax6.grid(alpha=0.3)
        ax7 = fig.add_subplot(gs[6, 0])
        regime_days = results["regime_days"]
        regime_labels = [regime_names[i] for i in range(5)]
        regime_counts = [regime_days[i] for i in range(5)]
        colors = [regime_colors[i] for i in range(5)]
        bars = ax7.bar(regime_labels, regime_counts, color=colors, alpha=0.7)
        ax7.set_ylabel("Days", fontsize=10, fontweight="bold")
        ax7.set_title("Regime Distribution", fontsize=12, fontweight="bold")
        ax7.tick_params(axis="x", rotation=45, labelsize=8)
        ax7.grid(axis="y", alpha=0.3)
        for bar in bars:
            height = bar.get_height()
            ax7.text(
                bar.get_x() + bar.get_width() / 2.0,
                height,
                f"{int(height)}",
                ha="center",
                va="bottom",
                fontsize=9,
            )
        ax8 = fig.add_subplot(gs[6, 1])
        ax8.axis("off")
        downside_returns = results_df["Daily_Return"][results_df["Daily_Return"] < 0]
        sortino = (
            results["annual_return"] / (downside_returns.std() * np.sqrt(252))
            if len(downside_returns) > 0 and downside_returns.std() > 0
            else 0
        )
        calmar = (
            results["annual_return"] / abs(results["max_drawdown"])
            if results["max_drawdown"] != 0
            else 0
        )
        metrics_text = f"\nFAMRS - FIXED VERSION\n{'=' * 50}\n\nPERFORMANCE:\n  Total Return:        {results['total_return']:>8.2%}\n  Annual Return:       {results['annual_return']:>8.2%}\n  Volatility:          {results['volatility']:>8.2%}\n  Sharpe Ratio:        {results['sharpe_ratio']:>8.2f}\n  Sortino Ratio:       {sortino:>8.2f}\n  Calmar Ratio:        {calmar:>8.2f}\n  Max Drawdown:        {results['max_drawdown']:>8.2%}\n\nLEVERAGE & COSTS:\n  Avg Leverage:        {results['avg_leverage']:>8.2f}x\n  Max Leverage:        {results['max_leverage']:>8.2f}x\n  Financing Costs:     ${results['total_financing_costs']:>8,.0f}\n\nBENCHMARK ({config.BENCHMARK_TICKER}):\n  Annual Return:       {results['benchmark_annual_return']:>8.2%}\n  Sharpe Ratio:        {results['benchmark_sharpe']:>8.2f}\n  Max Drawdown:        {results['benchmark_max_drawdown']:>8.2%}\n\nOUTPERFORMANCE:\n  Return Diff:         {results['annual_return'] - results['benchmark_annual_return']:>8.2%}\n\nTRADING:\n  Total Trades:        {results['total_trades']:>8d}\n  Stop Losses:         {results['stop_loss_count']:>8d}\n  Avg Positions:       {results['avg_positions']:>8.1f}\n        "
        ax8.text(
            0.05,
            0.95,
            metrics_text,
            transform=ax8.transAxes,
            fontsize=9,
            verticalalignment="top",
            fontfamily="monospace",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.3),
        )
        plt.suptitle("FAMRS - Fixed Execution & Leverage", fontsize=16, fontweight="bold", y=0.995)
        plt.tight_layout()
        plt.show()


def main():
    """Main execution function."""
    print("\n" + "=" * 80)
    print("FAMRS - FIXED VERSION")
    print("=" * 80)
    print("\nCRITICAL FIXES:")
    print("  1. Close[T-1] signal â†’ Open[T] execution")
    print("  2. Open-to-Open returns on execution days")
    print("  3. True leverage with financing costs")
    print("  4. Adjusted close prices for all calculations")
    print("  5. Benchmark aligned to same timing")
    print("  6. Stop-losses exit at Open[T+1] with slippage")
    config = StrategyConfig(
        BACKTEST_START_DATE="2013-01-01",
        BACKTEST_END_DATE="2018-12-31",
        INITIAL_CAPITAL=100000.0,
        BENCHMARK_TICKER="URTH",
        MAX_LEVERAGE=1.5,
        MARGIN_RATE_ANNUAL=0.06,
        STOP_LOSS_ENABLED=True,
        STOP_LOSS_SLIPPAGE=0.002,
        COMMISSION_RATE=0.0005,
        BID_ASK_SPREAD=0.0003,
        MARKET_IMPACT=0.0002,
    )
    print(f"\nBacktest: {config.BACKTEST_START_DATE} to {config.BACKTEST_END_DATE}")
    print(f"Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print(f"Max Leverage: {config.MAX_LEVERAGE:.1%}")
    print(f"Margin Rate: {config.MARGIN_RATE_ANNUAL:.2%}/year")
    try:
        fetcher = DataFetcher()
        data_dict = fetcher.fetch_all_data(config)
        features_dict = FeatureEngine.prepare_all_features(data_dict, config)
        detector = MultiAssetConsensusDetector(config)
        regime_predictions = detector.walk_forward_predict(features_dict)
        engine = BacktestEngine(config)
        results = engine.run_backtest(regime_predictions, data_dict)
        print("\n" + "=" * 80)
        print("GENERATING VISUALIZATIONS")
        print("=" * 80)
        Visualizer.create_dashboard(results, config)
        print("\n" + "=" * 80)
        print("EXPORTING RESULTS")
        print("=" * 80)
        results_df = results["results_df"]
        results_df.to_csv("famrs_fixed_portfolio_values.csv", index=False)
        print("âœ“ Portfolio values: famrs_fixed_portfolio_values.csv")
        if results["trades"]:
            trades_df = pd.DataFrame(results["trades"])
            trades_df.to_csv("famrs_fixed_trades_log.csv", index=False)
            print("âœ“ Trades log: famrs_fixed_trades_log.csv")
        if results["stop_loss_triggers"]:
            sl_df = pd.DataFrame(results["stop_loss_triggers"])
            sl_df.to_csv("famrs_fixed_stop_loss_triggers.csv", index=False)
            print("âœ“ Stop-losses: famrs_fixed_stop_loss_triggers.csv")
        regime_predictions.to_csv("famrs_fixed_regime_predictions.csv", index=False)
        print("âœ“ Regimes: famrs_fixed_regime_predictions.csv")
    except Exception as e:
        print(f"\nâœ— ERROR: {str(e)}")
        import traceback

        traceback.print_exc()
        return
    print("\n" + "=" * 80)
    print("FAMRS FIXED VERSION COMPLETE")
    print("=" * 80)
    print(f"\n  âœ“ Execution: Close[T-1] â†’ Open[T]")
    print(f"  âœ“ Leverage: {results['avg_leverage']:.2f}x avg, {results['max_leverage']:.2f}x max")
    print(f"  âœ“ Financing: ${results['total_financing_costs']:,.2f}")
    print(f"  âœ“ Annual Return: {results['annual_return']:.2%}")
    print(f"  âœ“ Sharpe: {results['sharpe_ratio']:.2f}")
    print(f"  âœ“ Max DD: {results['max_drawdown']:.2%}")
    print(
        f"  âœ“ vs {config.BENCHMARK_TICKER}: {results['annual_return'] - results['benchmark_annual_return']:+.2%}"
    )


if __name__ == "__main__":
    main()
